# CIFAR-10 Preprocessing V3 (Full Dataset)
- Full dataset: 50K train + 10K test
- Denoising → CLAHE → Sharpening on original 32×32
- Data Augmentation: 50K → 75K images
- Augmentations: Flip + Rotation + Crop + Brightness + Cutout

In [ ]:
%pip install scipy numpy matplotlib pillow scikit-learn opencv-python

## 1. Load Full Dataset (50K + 10K)

In [ ]:
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import cv2
import os

# Run from project root so all paths are uniform
if os.path.basename(os.getcwd()) == 'Preprocessing':
    os.chdir('..')

DATA_DIR = 'Dataset'

meta = scipy.io.loadmat(os.path.join(DATA_DIR, 'batches.meta.mat'))
label_names = [name[0] for name in meta['label_names'].flatten()]
print('Classes:', label_names)

train_data, train_labels = [], []
for i in range(1, 6):
    batch = scipy.io.loadmat(os.path.join(DATA_DIR, f'data_batch_{i}.mat'))
    train_data.append(batch['data'])
    train_labels.append(batch['labels'].flatten())

train_data = np.concatenate(train_data).reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.uint8)
train_labels = np.concatenate(train_labels)

test_batch = scipy.io.loadmat(os.path.join(DATA_DIR, 'test_batch.mat'))
test_data = test_batch['data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.uint8)
test_labels = test_batch['labels'].flatten()

print(f'Train: {train_data.shape} | Test: {test_data.shape}')

## 2. Denoising → CLAHE → Sharpening

In [ ]:
def preprocess_image(img):
    # Step 1: Gaussian Denoising
    denoised = cv2.GaussianBlur(img, (3, 3), 0.5)
    
    # Step 2: CLAHE on each channel
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    channels = cv2.split(denoised)
    clahe_channels = [clahe.apply(ch) for ch in channels]
    clahe_img = cv2.merge(clahe_channels)
    
    # Step 3: Unsharp Masking
    blurred = cv2.GaussianBlur(clahe_img, (3, 3), 1.0)
    sharpened = cv2.addWeighted(clahe_img, 1.5, blurred, -0.5, 0)
    
    return sharpened

def preprocess_batch(images, name=''):
    processed = np.zeros_like(images)
    for i in range(len(images)):
        processed[i] = preprocess_image(images[i])
        if (i + 1) % 10000 == 0:
            print(f'  {name} Processed {i + 1}/{len(images)}')
    return processed

print('Preprocessing training images (50K)...')
train_processed = preprocess_batch(train_data, 'Train:')
print('Preprocessing test images (10K)...')
test_processed = preprocess_batch(test_data, 'Test:')
print('Done!')

## 3. Before vs After Comparison

In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(train_data), 10, replace=False)

fig, axes = plt.subplots(2, 10, figsize=(20, 5))
fig.suptitle('Before vs After (Denoise + CLAHE + Sharpen)', fontsize=14, fontweight='bold')
for i, idx in enumerate(sample_idx):
    axes[0, i].imshow(train_data[idx]); axes[0, i].axis('off')
    axes[0, i].set_title(f'{label_names[train_labels[idx]]}\nOriginal', fontsize=8)
    axes[1, i].imshow(train_processed[idx]); axes[1, i].axis('off')
    axes[1, i].set_title('Processed', fontsize=8)
plt.tight_layout()
plt.savefig('Figure/before_after_v3.png', dpi=150)
plt.show()

## 4. Data Augmentation (50K → 75K)

In [ ]:
def cutout(img, size=8):
    """Randomly mask out a square patch"""
    h, w = img.shape[:2]
    y = np.random.randint(0, h)
    x = np.random.randint(0, w)
    y1 = max(0, y - size // 2)
    y2 = min(h, y + size // 2)
    x1 = max(0, x - size // 2)
    x2 = min(w, x + size // 2)
    result = img.copy()
    result[y1:y2, x1:x2] = 0
    return result

def augment_image(img):
    aug = img.copy()
    
    # Random horizontal flip (50%)
    if np.random.random() > 0.5:
        aug = np.fliplr(aug)
    
    # Random rotation (-15 to +15 degrees)
    angle = np.random.uniform(-15, 15)
    h, w = aug.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    aug = cv2.warpAffine(aug, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    
    # Random crop (pad 4px, crop back to 32x32)
    padded = np.pad(aug, ((4, 4), (4, 4), (0, 0)), mode='reflect')
    x = np.random.randint(0, 8)
    y = np.random.randint(0, 8)
    aug = padded[y:y+32, x:x+32]
    
    # Random brightness (0.8 to 1.2)
    factor = np.random.uniform(0.8, 1.2)
    aug = np.clip(aug * factor, 0, 255).astype(np.uint8)
    
    # Cutout (50% chance)
    if np.random.random() > 0.5:
        aug = cutout(aug, size=10)
    
    return aug

# Generate 25,000 augmented images (balanced: 2,500 per class)
NUM_AUGMENTED = 25000
np.random.seed(42)
samples_per_class = NUM_AUGMENTED // 10

aug_images = []
aug_labels = []

for cls in range(10):
    cls_indices = np.where(train_labels == cls)[0]
    chosen = np.random.choice(cls_indices, samples_per_class, replace=True)
    for idx in chosen:
        aug_images.append(augment_image(train_processed[idx]))
        aug_labels.append(cls)
    print(f'  Augmented {label_names[cls]}: {samples_per_class} images')

aug_images = np.array(aug_images)
aug_labels = np.array(aug_labels)

# Combine original + augmented
train_final = np.concatenate([train_processed, aug_images])
labels_final = np.concatenate([train_labels, aug_labels])

# Shuffle
shuffle_idx = np.random.permutation(len(train_final))
train_final = train_final[shuffle_idx]
labels_final = labels_final[shuffle_idx]

print(f'\nOriginal:  {train_processed.shape[0]:,} images')
print(f'Augmented: {aug_images.shape[0]:,} images')
print(f'Total:     {train_final.shape[0]:,} images')

## 5. Show Augmented Samples

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(20, 7))
fig.suptitle('Original → Processed → Augmented', fontsize=14, fontweight='bold')
np.random.seed(99)
for i in range(10):
    idx = np.random.choice(np.where(train_labels == i)[0])
    axes[0, i].imshow(train_data[idx]); axes[0, i].axis('off')
    axes[0, i].set_title(f'{label_names[i]}\nOriginal', fontsize=8)
    axes[1, i].imshow(train_processed[idx]); axes[1, i].axis('off')
    axes[1, i].set_title('Processed', fontsize=8)
    axes[2, i].imshow(augment_image(train_processed[idx])); axes[2, i].axis('off')
    axes[2, i].set_title('Augmented', fontsize=8)
plt.tight_layout()
plt.savefig('Figure/augmentation_samples_v3.png', dpi=150)
plt.show()

## 6. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, labels, title in zip(axes, [train_labels, labels_final], ['Original (50K)', 'After Augmentation (75K)']):
    counts = np.bincount(labels)
    ax.bar(range(10), counts, color='steelblue')
    ax.set_xticks(range(10)); ax.set_xticklabels(label_names, rotation=45, ha='right')
    ax.set_ylabel('Count'); ax.set_title(title)
    for j, c in enumerate(counts):
        ax.text(j, c + 50, str(c), ha='center', fontsize=8)
plt.tight_layout()
plt.savefig('Figure/class_distribution_v3.png', dpi=150)
plt.show()

## 7. Normalize & Standardize

In [ ]:
train_norm = train_final.astype(np.float32) / 255.0
test_norm = test_processed.astype(np.float32) / 255.0

channel_mean = train_norm.mean(axis=(0, 1, 2))
channel_std = train_norm.std(axis=(0, 1, 2))
print(f'Channel means (R,G,B): {channel_mean}')
print(f'Channel stds  (R,G,B): {channel_std}')

train_std = (train_norm - channel_mean) / channel_std
test_std = (test_norm - channel_mean) / channel_std

print(f'\nTrain - min: {train_std.min():.3f}, max: {train_std.max():.3f}, mean: {train_std.mean():.6f}')
print(f'Test  - min: {test_std.min():.3f}, max: {test_std.max():.3f}, mean: {test_std.mean():.6f}')

## 8. Train / Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    train_std, labels_final,
    test_size=0.1, random_state=42, stratify=labels_final
)

X_test = test_std
y_test = test_labels

print(f'Training set:   {X_train.shape} | Labels: {y_train.shape}')
print(f'Validation set: {X_val.shape}  | Labels: {y_val.shape}')
print(f'Test set:       {X_test.shape} | Labels: {y_test.shape}')

## 9. Save Preprocessed Data

In [ ]:
out_path = 'Preprocessing/cifar10_v3_preprocessed.npz'
np.savez_compressed(out_path,
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test,
    channel_mean=channel_mean, channel_std=channel_std,
    label_names=label_names
)
print(f'Saved: {out_path} ({os.path.getsize(out_path) / (1024**2):.1f} MB)')

## Summary

| Step | Details |
|---|---|
| Dataset | **Full: 50K train + 10K test** |
| Resolution | Original 32×32 |
| Denoising | Gaussian blur (3×3, σ=0.5) |
| Contrast | CLAHE (clipLimit=2.0, tile=4×4) |
| Sharpening | Unsharp masking (1.5x) |
| Augmentation | Flip + Rotation + Crop + Brightness + **Cutout** |
| Total images | **75K** (50K original + 25K augmented) |
| Normalize | Per-channel standardize |
| Split | **67.5K train / 7.5K val / 10K test** |